# IPN variable length — Step 4 smoke test

Notebook kiểm tra Dataset, dynamic padding, cached velocity và Transformer padding mask trên cache `variable_v1` ở Drive. Nó chỉ smoke-test clip `<=256`; clip dài được đếm và để lại cho sliding window bước 5. Notebook không train và không tạo metric phân loại.

In [ ]:
from pathlib import Path
import datetime as dt
import json
import subprocess
import sys
import torch
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_BRANCH = 'HuyenHoang'
REPO_ROOT = Path('/content/dynamic_hand_gesture_recognition')
DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
CACHE_ROOT = DATA_ROOT / 'keypoint_tensor_cache'
MAX_FRAMES = 256
EXPECTED = {'train': 2443, 'val': 589, 'test': 1101}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def run(argv, cwd=None):
    print('$', ' '.join(map(str, argv)))
    subprocess.run([str(value) for value in argv], cwd=cwd, check=True)


In [ ]:
if (REPO_ROOT / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_ROOT)
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_ROOT)
else:
    run(['git', 'clone', '--filter=blob:none', '--branch', REPO_BRANCH, REPO_URL, REPO_ROOT])
sys.path[:0] = [str(REPO_ROOT / 'final/code'), str(REPO_ROOT / 'final/code/neural_only_experiments')]
from src_neurosymbolic.variable_keypoint_cache import load_variable_cache
from neural_only_experiments.variable_dataset import VariableKeypointDataset, collate_variable_batch, variable_cache_path
from models import NeuralOnlyClassifier
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
print({'commit': commit, 'device': str(DEVICE)})
smoke_report_path = CACHE_ROOT / 'variable_v1' / 'STEP4_SMOKE_REPORT.json'
smoke_report_path.parent.mkdir(parents=True, exist_ok=True)
smoke_tmp = smoke_report_path.with_suffix('.json.tmp')
smoke_tmp.write_text(json.dumps({
    'status': 'RUNNING',
    'timestamp_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'repo_commit': commit,
}, indent=2) + '\n', encoding='utf-8')
smoke_tmp.replace(smoke_report_path)

In [ ]:
build_report = CACHE_ROOT / 'variable_v1' / 'CACHE_BUILD_REPORT.json'
assert build_report.is_file(), 'Chạy notebook bước 3 trước'
build_status = json.loads(build_report.read_text(encoding='utf-8'))
assert build_status.get('status') == 'PASS', build_status

split_audit = {}
eligible_indices = {}
padding_test_indices = {}
for split, expected in EXPECTED.items():
    manifest = DATA_ROOT / f'{split}.txt'
    dataset = VariableKeypointDataset(manifest, CACHE_ROOT, split, max_frames=MAX_FRAMES)
    assert len(dataset) == expected
    short, shorter, long = [], [], []
    for index, (source, _) in enumerate(dataset.samples):
        payload = load_variable_cache(variable_cache_path(source, CACHE_ROOT))
        if payload['length'] <= MAX_FRAMES:
            short.append(index)
            if payload['length'] < MAX_FRAMES:
                shorter.append(index)
        else:
            long.append(index)
    eligible_indices[split] = short
    padding_test_indices[split] = shorter
    split_audit[split] = {'clips': expected, 'at_most_256': len(short), 'over_256': len(long)}
print(json.dumps(split_audit, indent=2))

In [ ]:
from torch.utils.data import DataLoader, Subset

train_dataset = VariableKeypointDataset(DATA_ROOT / 'train.txt', CACHE_ROOT, 'train', max_frames=MAX_FRAMES)
short_train = Subset(train_dataset, eligible_indices['train'][:32])
loader = DataLoader(short_train, batch_size=8, shuffle=False, collate_fn=collate_variable_batch)
batch = next(iter(loader))
model = NeuralOnlyClassifier(
    backbone='transformer', num_classes=13, num_frames=MAX_FRAMES,
    d_model=192, layers=4, heads=8, dropout=0.0,
    use_motion_features=True, pooling='masked_mean',
).to(DEVICE).eval()
with torch.no_grad():
    logits = model(
        batch['keypoints'].to(DEVICE),
        padding_mask=batch['padding_mask'].to(DEVICE),
        velocity=batch['velocity'].to(DEVICE),
    )
assert logits.shape == (len(batch['lengths']), 13)
assert torch.isfinite(logits).all()
assert torch.equal(batch['padding_mask'].sum(1), batch['keypoints'].shape[1] - batch['lengths'])
print({'batch_shape': list(batch['keypoints'].shape), 'lengths': batch['lengths'].tolist(), 'logits_finite': True})

In [ ]:
assert padding_test_indices['train'], 'Không có train clip ngắn hơn 256 để test padding'
item = train_dataset[padding_test_indices['train'][0]]
single = collate_variable_batch([item])
extra = min(7, MAX_FRAMES - int(item['length']))
assert extra > 0, 'Chọn sample ngắn hơn 256 để test padding invariance'
padded_keypoints = torch.nn.functional.pad(single['keypoints'], (0, 0, 0, 0, 0, extra), value=999.0)
padded_velocity = torch.nn.functional.pad(single['velocity'], (0, 0, 0, 0, 0, extra), value=999.0)
padded_mask = torch.nn.functional.pad(single['padding_mask'], (0, extra), value=True)
with torch.no_grad():
    base = model(single['keypoints'].to(DEVICE), padding_mask=single['padding_mask'].to(DEVICE), velocity=single['velocity'].to(DEVICE))
    padded = model(padded_keypoints.to(DEVICE), padding_mask=padded_mask.to(DEVICE), velocity=padded_velocity.to(DEVICE))
max_delta = float((base - padded).abs().max().cpu())
assert torch.allclose(base, padded, atol=1e-5, rtol=1e-5), max_delta

report = {
    'status': 'PASS',
    'timestamp_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'repo_commit': commit,
    'max_frames': MAX_FRAMES,
    'splits': split_audit,
    'smoke_batch_shape': list(batch['keypoints'].shape),
    'padding_invariance_max_logit_delta': max_delta,
    'note': 'Clips over 256 intentionally await Step 5 windowing.',
}
smoke_tmp.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
smoke_tmp.replace(smoke_report_path)
print(json.dumps(report, indent=2))
print('Saved:', smoke_report_path)